In [1]:
# Cell 1 — Install required libraries
!pip install langchain langchain-google-genai langchain-community faiss-cpu wikipedia gradio -q

print("All libraries ready!")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
All libraries ready!


In [2]:
# Cell 2 — Load API key securely
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
print("API key loaded securely!")

API key loaded securely!


In [3]:
# Cell 3 — Test connection to Gemini (corrected model name)
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

response = llm.invoke("Say hello and tell me one fun fact about Pakistan in one sentence.")
print(response.content)

Hello! Pakistan is home to the world's highest ATM, located at an elevation of 15,300 feet in the Khunjerab Pass.


In [4]:
# Cell 3b — List available models for your API key
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [17]:
# Cell 4 — Load document content (reliable, no external API dependency)
from langchain_core.documents import Document

ai_content = """
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games such as chess and Go. Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.

Machine learning is a subset of AI that enables systems to learn and improve from experience without being explicitly programmed. Deep learning, a further subset of machine learning, uses neural networks with many layers to analyze various factors of data.

Natural language processing (NLP) gives machines the ability to read, understand, and derive meaning from human languages. It powers applications like translation, chatbots, and voice assistants.

Computer vision allows machines to interpret and make decisions based on visual data from the world, such as images and videos. It is used in facial recognition, medical image analysis, and autonomous vehicles.

AI raises important ethical questions around bias, privacy, job displacement, and the need for transparent and accountable systems. Researchers and policymakers continue to work on frameworks for responsible AI development.

The history of AI dates back to the 1950s when researchers first began exploring whether machines could think. Early AI research focused on problem solving and symbolic methods. The field has gone through several cycles of optimism followed by reduced funding, often called AI winters, before the recent resurgence driven by deep learning and large language models.
"""

documents = [Document(page_content=ai_content, metadata={"title": "Artificial Intelligence Overview"})]

print(f"Loaded {len(documents)} document(s)")
print()
print("Document title:", documents[0].metadata['title'])
print()
print("Total characters in document:", len(documents[0].page_content))

Loaded 1 document(s)

Document title: Artificial Intelligence Overview

Total characters in document: 2052


In [18]:
# Cell 5 — Split document into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # each chunk = ~500 characters
    chunk_overlap=50     # chunks slightly overlap to preserve context
)

chunks = text_splitter.split_documents(documents)

print(f"Document split into {len(chunks)} chunks")
print()
print("First chunk:")
print(chunks[0].page_content)
print()
print("Second chunk:")
print(chunks[1].page_content)

Document split into 5 chunks

First chunk:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.

Second chunk:
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games such as chess and Go. Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.


In [19]:
# Cell 6 — Create embeddings and build vector store
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# Embeddings model — converts text into meaning-vectors
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Build the vector store from our chunks
vectorstore = FAISS.from_documents(chunks, embeddings)

print("Vector store created!")
print(f"Total vectors stored: {vectorstore.index.ntotal}")

# Quick test — search for something related, see if it finds the right chunk
test_query = "What are some real world uses of AI?"
results = vectorstore.similarity_search(test_query, k=2)

print(f"\nTop 2 chunks matching: '{test_query}'\n")
for i, doc in enumerate(results):
    print(f"Match {i+1}:")
    print(doc.page_content)
    print("-" * 60)

Vector store created!
Total vectors stored: 5

Top 2 chunks matching: 'What are some real world uses of AI?'

Match 1:
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games such as chess and Go. Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.
------------------------------------------------------------
Match 2:
Computer vision allows machines to interpret and make decisions based on visual data from the world, such as images and videos. It is used in facial recognition, medical image analysis, and autonomous vehicles.

AI raises important ethical questions around bias, privacy, job displacement, and the need for transparent and accountable systems. Researchers and policymakers continue to work on frameworks for responsible AI development.
------------------------------------------------------------


In [20]:
# Cell 6b — List available embedding models
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [27]:
# Cell 7 — Build a lean RAG chain (fewer API calls per question)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})  # only top 2 chunks now

chat_history = []  # manual, simple memory — just a list of (question, answer) pairs

def ask_question(question):
    # Retrieve relevant chunks manually (1 embedding call)
    relevant_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    # Build history string for context
    history_text = ""
    for q, a in chat_history[-2:]:  # only last 2 exchanges, keep it light
        history_text += f"Previous Q: {q}\nPrevious A: {a}\n\n"

    # Single prompt combining everything — only 1 LLM call, 1 embedding call total
    prompt = f"""Answer the question using ONLY the context below. Be concise.

{history_text}Context:
{context}

Question: {question}
Answer:"""

    response = llm.invoke(prompt)
    answer = response.content

    chat_history.append((question, answer))
    return answer

print("Lean RAG chatbot ready!")

Lean RAG chatbot ready!


In [22]:
# Cell 7b — Ensure core langchain package is installed
!pip install langchain -q --upgrade

print("Done! Now restart and re-run from Cell 7")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 11.0 MB/s eta 0:00:00
Done! Now restart and re-run from Cell 7


In [23]:
# Diagnostic — check langchain installation
!pip show langchain
print()
!pip list | grep langchain

Name: langchain
Version: 1.3.10
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 

langchain                                1.3.10
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.4.8
langchain-google-genai                   4.2.5
langchain-protocol                       0.0.18
langchain-text-splitters                 1.1.2


In [28]:
# Cell 8 — Have a conversation (lean version, fewer API calls)
import time

question1 = "What is artificial intelligence?"
answer1 = ask_question(question1)
print("Q1:", question1)
print("A1:", answer1)

print("\nWaiting 20 seconds...\n")
time.sleep(20)

question2 = "What are some applications of it?"
answer2 = ask_question(question2)
print("Q2:", question2)
print("A2:", answer2)

print("\nWaiting 20 seconds...\n")
time.sleep(20)

question3 = "Can you give an example from what you just mentioned?"
answer3 = ask_question(question3)
print("Q3:", question3)
print("A3:", answer3)

Q1: What is artificial intelligence?
A1: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research that develops and studies methods and software enabling machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.

Waiting 20 seconds...

Q2: What are some applications of it?
A2: Some applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games, and generative AI for images, audio, and videos. It also powers translation.

Waiting 20 seconds...

Q3: Can you give an example from what you just mentioned?
A3: Advanced web search engines.


In [14]:
# Diagnostic — test embeddings directly with a longer wait
import time

print("Waiting 30 seconds to let any rate limits reset...")
time.sleep(30)

try:
    test_embed = embeddings.embed_query("test sentence")
    print("Embeddings working! Vector length:", len(test_embed))
except Exception as e:
    print("Still failing:", e)

Waiting 30 seconds to let any rate limits reset...
Embeddings working! Vector length: 3072


In [15]:
# Diagnostic — see the actual error, not just "failed"
try:
    response1 = qa_chain.invoke({"question": "What is AI?"})
    print("Success:", response1['answer'])
except Exception as e:
    print("ACTUAL ERROR:")
    print(type(e).__name__)
    print(str(e))


ACTUAL ERROR:
GoogleGenerativeAIError
Error embedding content: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}


In [16]:
# Check exact rate limit error details
import time
time.sleep(5)

try:
    test = embeddings.embed_query("hello")
    print("Works fine right now")
except Exception as e:
    print("Error details:")
    print(str(e))

Works fine right now


In [29]:
# Cell 9 — Deploy chatbot with Gradio
import gradio as gr
import time

def chatbot_response(message, history):
    try:
        answer = ask_question(message)
        return answer
    except Exception as e:
        return "Sorry, I'm experiencing high traffic right now. Please try again in a moment."

demo = gr.ChatInterface(
    fn=chatbot_response,
    title="🤖 AI Knowledge Chatbot (RAG-powered)",
    description="Ask me anything about Artificial Intelligence! I remember our conversation and answer based on a knowledge document, not just guessing.",
    examples=[
        "What is artificial intelligence?",
        "What is machine learning?",
        "Tell me about the history of AI",
        "What ethical issues does AI raise?"
    ]
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b45a9e0d89efdf2a32.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
